In [21]:
import os
import numpy as np
import pandas as pd
import torch
from itertools import product
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split, cross_val_predict
from sklearn.metrics import balanced_accuracy_score, classification_report
from sklearn.utils import resample
from sklearn.utils.class_weight import compute_class_weight
from sklearn.linear_model import LogisticRegression
from sklearn.feature_extraction.text import TfidfVectorizer

from transformers import AutoTokenizer,AutoModelForSequenceClassification,TrainingArguments,Trainer,EarlyStoppingCallback

from cleanlab.filter import find_label_issues

os.environ["CUDA_VISIBLE_DEVICES"] = "0"
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU mode")

NVIDIA GeForce RTX 3050 6GB Laptop GPU


In [3]:

data_raw = pd.read_csv('dataset2/train_clean_v2.csv')
label_encoder = LabelEncoder()
data_raw['label_id'] = label_encoder.fit_transform(data_raw['label'])
num_labels = len(label_encoder.classes_)
id2label   = {i: l for i, l in enumerate(label_encoder.classes_)}
label2id   = {l: i for i, l in id2label.items()}

print('Shape    :', data_raw.shape)
print('Kelas    :', list(label_encoder.classes_))
print('Dist     :')
print(data_raw['label'].value_counts())

Shape    : (5000, 3)
Kelas    : ['Anggaran', 'Distribusi', 'Ekonomi', 'Kualitas Pangan', 'Lainnya', 'Politik', 'Sasaran Penerima', 'Tata Kelola']
Dist     :
label
Kualitas Pangan     1247
Politik              792
Anggaran             727
Lainnya              638
Tata Kelola          511
Sasaran Penerima     507
Distribusi           433
Ekonomi              145
Name: count, dtype: int64


In [4]:
vec = TfidfVectorizer(max_features=10000)
X_tfidf = vec.fit_transform(data_raw['full_text'])
y = data_raw['label_id'].values

clf = LogisticRegression(max_iter=1000, class_weight='balanced')
pred_probs_tfidf = cross_val_predict(clf, X_tfidf, y, cv=5, method='predict_proba')

label_issues = find_label_issues(
    labels=y,
    pred_probs=pred_probs_tfidf,
    return_indices_ranked_by='self_confidence'
)

data_raw['is_noisy'] = False
data_raw.loc[label_issues, 'is_noisy'] = True

print(f'Noisy label terdeteksi: {data_raw["is_noisy"].sum()} dari {len(data_raw)}')
print(f'Data bersih           : {(~data_raw["is_noisy"]).sum()}')

Noisy label terdeteksi: 1237 dari 5000
Data bersih           : 3763


## 1. Oversample Kelas Minoritas
> Kelas Ekonomi dan distribusi di ambil setiap katanya untuk di resamble menjadi kalimat baru dengan label yang sama. Tujuannya untuk membuat data tidak terlalu kecil

In [5]:
RANDOM_SEED = 42

# Soft label dari TF-IDF (seluruh data, sebelum filter)
ALPHA = 0.7
one_hot_raw  = np.eye(num_labels)[data_raw['label_id'].values]
soft_raw     = ALPHA * one_hot_raw + (1 - ALPHA) * pred_probs_tfidf

# Filter noisy dulu sebelum oversample
clean_mask   = ~data_raw['is_noisy'].values
data_clean   = data_raw[clean_mask].reset_index(drop=True)
soft_clean   = soft_raw[clean_mask]

print(f'Data bersih: {len(data_clean)}')
print(data_clean['label'].value_counts())

# Oversample kelas minoritas
OVERSAMPLE_CONFIG = {
    'Ekonomi'   : 450,
    'Distribusi': 650,
}

parts     = []
soft_parts = []

for label_name, target_n in OVERSAMPLE_CONFIG.items():
    mask   = data_clean['label'] == label_name
    df_cls = data_clean[mask]
    sl_cls = soft_clean[mask.values]

    idx_resampled = resample(
        np.arange(len(df_cls)),
        replace=True,
        n_samples=target_n,
        random_state=RANDOM_SEED
    )
    parts.append(df_cls.iloc[idx_resampled].reset_index(drop=True))
    soft_parts.append(sl_cls[idx_resampled])

# Kelas lain tidak di-oversample
mask_others = ~data_clean['label'].isin(OVERSAMPLE_CONFIG.keys())
parts.append(data_clean[mask_others].reset_index(drop=True))
soft_parts.append(soft_clean[mask_others.values])

# Gabung & shuffle
data         = pd.concat(parts, ignore_index=True)
soft_labels  = np.vstack(soft_parts)

shuffle_idx  = np.random.RandomState(RANDOM_SEED).permutation(len(data))
data         = data.iloc[shuffle_idx].reset_index(drop=True)
soft_labels  = soft_labels[shuffle_idx]

print('\nDistribusi setelah oversample:')
print(data['label'].value_counts())
print('\nSoft label shape:', soft_labels.shape)

Data bersih: 3763
label
Kualitas Pangan     898
Politik             569
Anggaran            562
Lainnya             509
Sasaran Penerima    386
Tata Kelola         383
Distribusi          343
Ekonomi             113
Name: count, dtype: int64

Distribusi setelah oversample:
label
Kualitas Pangan     898
Distribusi          650
Politik             569
Anggaran            562
Lainnya             509
Ekonomi             450
Sasaran Penerima    386
Tata Kelola         383
Name: count, dtype: int64

Soft label shape: (4407, 8)


## 2. Train/Val Split

In [6]:
train_df, valid_df, soft_train, soft_valid = train_test_split(
    data,
    soft_labels,
    test_size=0.2,
    stratify=data['label_id'],
    random_state=RANDOM_SEED
)

train_df = train_df.reset_index(drop=True)
valid_df = valid_df.reset_index(drop=True)

print(f'Train : {len(train_df)} | Val: {len(valid_df)}')
print('\nDist val:')
print(valid_df['label'].value_counts())

Train : 3525 | Val: 882

Dist val:
label
Kualitas Pangan     180
Distribusi          130
Politik             114
Anggaran            112
Lainnya             102
Ekonomi              90
Sasaran Penerima     77
Tata Kelola          77
Name: count, dtype: int64


## 3. Tokenisasi

In [7]:
MODEL     = 'xlm-roberta-base'
tokenizer = AutoTokenizer.from_pretrained(MODEL)

def tokenize(text_series):
    return tokenizer(
        text_series.tolist(),
        padding=True,
        truncation=True,
        max_length=128
    )

train_encodings = tokenize(train_df['full_text'])
valid_encodings = tokenize(valid_df['full_text'])

## 4. Dataset Class

In [9]:
class SoftLabelDataset(torch.utils.data.Dataset):
    """Dataset yang support hard label (int) dan soft label (float array)"""
    def __init__(self, encodings, labels, soft_labels=None):
        self.encodings   = encodings
        self.labels      = labels
        self.soft_labels = soft_labels

    def __getitem__(self, idx):
        item = {
            key: torch.tensor(val[idx])
            for key, val in self.encodings.items()
        }
        item['labels'] = torch.tensor(self.labels[idx], dtype=torch.long)
        if self.soft_labels is not None:
            item['soft_labels'] = torch.tensor(self.soft_labels[idx], dtype=torch.float)
        return item

    def __len__(self):
        return len(self.labels)


train_dataset = SoftLabelDataset(
    train_encodings,
    train_df['label_id'].tolist(),
    soft_labels=soft_train
)
valid_dataset = SoftLabelDataset(
    valid_encodings,
    valid_df['label_id'].tolist()
)

print(f'Train dataset: {len(train_dataset)}')
print(f'Val dataset  : {len(valid_dataset)}')

Train dataset: 3525
Val dataset  : 882


## 5. Class Weights + Boost Ekonomi & NoisyAwareTrainer

In [10]:
class_weights_np = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(train_df['label_id']),
    y=train_df['label_id']
)

# Boost Ekonomi 2x — masih sering missed meski sudah balanced + oversample
ekonomi_idx = label2id['Ekonomi']
print(f'Weight Ekonomi sebelum boost: {class_weights_np[ekonomi_idx]:.3f}')
class_weights_np[ekonomi_idx] *= 2.0
print(f'Weight Ekonomi setelah boost: {class_weights_np[ekonomi_idx]:.3f}')

class_weights = torch.tensor(class_weights_np, dtype=torch.float)
print('\nSemua class weights:', dict(zip(label_encoder.classes_, class_weights_np.round(3))))


class NoisyAwareTrainer(Trainer):
    def __init__(self, class_weights, noise_percentile=80, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights    = class_weights
        self.noise_percentile = noise_percentile

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels      = inputs['labels']
        soft_labels = inputs.get('soft_labels', None)

        outputs = model(
            input_ids=inputs['input_ids'],
            attention_mask=inputs['attention_mask']
        )
        logits = outputs.logits

        if soft_labels is not None:
            # KL-divergence untuk soft label
            log_probs = torch.nn.functional.log_softmax(logits, dim=-1)
            per_sample_loss = torch.nn.functional.kl_div(
                log_probs, soft_labels, reduction='none'
            ).sum(dim=-1)
        else:
            # Weighted CrossEntropy untuk val
            loss_fct = torch.nn.CrossEntropyLoss(
                weight=self.class_weights.to(model.device),
                reduction='none',
                label_smoothing=0.1
            )
            per_sample_loss = loss_fct(
                logits.view(-1, model.config.num_labels),
                labels.view(-1)
            )

        # Buang sampel dengan loss tertinggi (kemungkinan masih noisy)
        threshold = torch.quantile(per_sample_loss, self.noise_percentile / 100.0)
        mask      = per_sample_loss <= threshold
        loss      = per_sample_loss[mask].mean()

        return (loss, outputs) if return_outputs else loss

Weight Ekonomi sebelum boost: 1.224
Weight Ekonomi setelah boost: 2.448

Semua class weights: {'Anggaran': np.float64(0.979), 'Distribusi': np.float64(0.847), 'Ekonomi': np.float64(2.448), 'Kualitas Pangan': np.float64(0.614), 'Lainnya': np.float64(1.083), 'Politik': np.float64(0.968), 'Sasaran Penerima': np.float64(1.426), 'Tata Kelola': np.float64(1.44)}


## 6. Training

In [16]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {'balanced_accuracy': balanced_accuracy_score(labels, preds)}


model = AutoModelForSequenceClassification.from_pretrained(
    MODEL,
    num_labels=num_labels,
    id2label=id2label,
    label2id=label2id
)

training_args = TrainingArguments(
    output_dir='./results',
    eval_strategy='epoch',
    save_strategy='epoch',
    learning_rate=2e-5,
    warmup_ratio=0.1,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=10,
    weight_decay=0.05,
    lr_scheduler_type='cosine',
    load_best_model_at_end=True,
    metric_for_best_model='balanced_accuracy',
    greater_is_better=True,
    report_to='none',
    seed=RANDOM_SEED
)

trainer = NoisyAwareTrainer(
    class_weights=class_weights,
    noise_percentile=80,
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=valid_dataset,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)

trainer.train()

Loading weights: 100%|██████████| 197/197 [00:00<00:00, 2329.07it/s]
[transformers] XLMRobertaForSequenceClassification LOAD REPORT from: xlm-roberta-base
Key                         | Status     | 
----------------------------+------------+-
lm_head.dense.weight        | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
classifier.dense.weight     | MISSING    | 
classifier.out_proj.bias    | MISSING    | 
classifier.dense.bias       | MISSING    | 
classifier.out_proj.weight  | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
[transformers] warmup_rat

Epoch,Training Loss,Validation Loss,Balanced Accuracy
1,No log,1.291258,0.496768
2,No log,0.936656,0.650620
3,1.407738,0.753816,0.740445
4,1.407738,0.719027,0.760132
5,0.709093,0.731750,0.754276
6,0.709093,0.721264,0.769715
7,0.584803,0.723204,0.769146
8,0.584803,0.703539,0.767475


Writing model shards: 100%|██████████| 1/1 [00:02<00:00,  2.94s/it]


TrainOutput(global_step=1768, training_loss=0.8502395595360666, metrics={'train_runtime': 1746.6673, 'train_samples_per_second': 20.181, 'train_steps_per_second': 1.265, 'total_flos': 1333304874201600.0, 'train_loss': 0.8502395595360666, 'epoch': 8.0})

## 7. Evaluasi

In [17]:
pred    = trainer.predict(valid_dataset)
logits  = pred.predictions
probs   = torch.softmax(torch.tensor(logits), dim=-1).numpy()
preds   = np.argmax(probs, axis=-1)
true_labels = pred.label_ids

bal_acc = balanced_accuracy_score(true_labels, preds)
print(f'Balanced Accuracy: {bal_acc:.4f}')
print()
print(classification_report(
    true_labels, preds,
    target_names=label_encoder.classes_
))

Balanced Accuracy: 0.7697

                  precision    recall  f1-score   support

        Anggaran       0.84      0.93      0.88       112
      Distribusi       0.74      0.96      0.84       130
         Ekonomi       0.95      0.88      0.91        90
 Kualitas Pangan       0.80      0.84      0.82       180
         Lainnya       0.75      0.53      0.62       102
         Politik       0.80      0.66      0.72       114
Sasaran Penerima       0.70      0.75      0.72        77
     Tata Kelola       0.67      0.61      0.64        77

        accuracy                           0.79       882
       macro avg       0.78      0.77      0.77       882
    weighted avg       0.79      0.79      0.78       882



## 8. Threshold Tuning Per Kelas

Fokus ke Ekonomi dengan opsi threshold yang lebih agresif.

In [19]:
def predict_with_threshold(probs, thresholds):
    adjusted = probs / np.array(thresholds)
    return np.argmax(adjusted, axis=-1)


baseline = balanced_accuracy_score(true_labels, np.argmax(probs, axis=-1))
print(f'Baseline (no tuning): {baseline:.4f}')

best_score  = baseline
best_thresh = [1.0] * num_labels

weak_classes = [
    label2id['Ekonomi'],          # prioritas utama
    label2id['Distribusi'],
    label2id['Tata Kelola'],
    label2id['Lainnya'],
]

thresh_options_per_class = [
    [0.2, 0.3, 0.4, 0.5, 0.6],   # Ekonomi — agresif
    [0.6, 0.7, 0.8, 0.9, 1.0],   # Distribusi
    [0.6, 0.7, 0.8, 0.9, 1.0],   # Tata Kelola
    [0.6, 0.7, 0.8, 0.9, 1.0],   # Lainnya
]

for combo in product(*thresh_options_per_class):
    thresholds = [1.0] * num_labels
    for i, cls_idx in enumerate(weak_classes):
        thresholds[cls_idx] = combo[i]

    preds_tuned = predict_with_threshold(probs, thresholds)
    score = balanced_accuracy_score(true_labels, preds_tuned)

    if score > best_score:
        best_score  = score
        best_thresh = thresholds

print(f'Best score setelah tuning : {best_score:.4f}')
print(f'Best thresholds           : {[round(t, 2) for t in best_thresh]}')
print(f'Kelas                     : {list(label_encoder.classes_)}')

Baseline (no tuning): 0.7697
Best score setelah tuning : 0.7697
Best thresholds           : [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]
Kelas                     : ['Anggaran', 'Distribusi', 'Ekonomi', 'Kualitas Pangan', 'Lainnya', 'Politik', 'Sasaran Penerima', 'Tata Kelola']


In [20]:
final_preds = predict_with_threshold(probs, best_thresh)

print('Classification Report (setelah threshold tuning):')
print(classification_report(
    true_labels, final_preds,
    target_names=list(label_encoder.classes_)
))

Classification Report (setelah threshold tuning):
                  precision    recall  f1-score   support

        Anggaran       0.84      0.93      0.88       112
      Distribusi       0.74      0.96      0.84       130
         Ekonomi       0.95      0.88      0.91        90
 Kualitas Pangan       0.80      0.84      0.82       180
         Lainnya       0.75      0.53      0.62       102
         Politik       0.80      0.66      0.72       114
Sasaran Penerima       0.70      0.75      0.72        77
     Tata Kelola       0.67      0.61      0.64        77

        accuracy                           0.79       882
       macro avg       0.78      0.77      0.77       882
    weighted avg       0.79      0.79      0.78       882



## 9. Export OOF

In [ ]:
# # Predict dari data_raw (asli, tanpa oversample) biar shape match saat ensemble
# print('Predicting seluruh data asli untuk OOF...')

# all_encodings = tokenize(data_raw['full_text'])

# full_dataset = SoftLabelDataset(
#     {key: list(val) for key, val in all_encodings.items()},
#     data_raw['label_id'].tolist()
# )

# full_output = trainer.predict(full_dataset)
# full_probs  = torch.softmax(
#     torch.tensor(full_output.predictions), dim=-1
# ).numpy()

# print('OOF shape:', full_probs.shape)   # harus (N_raw, 8)
# print('Sum per baris (harus ~1):', full_probs[0].sum().round(4))

# # Simpan .npy
# # Opsional, kalau gak dipake, di # aja
# np.save('dataset2/xlmr_oof_proba.npy', full_probs)
# print('OOF disimpan: dataset2/xlmr_oof_proba.npy')

# # Simpan threshold terbaik
# np.save('dataset2/xlmr_best_thresholds.npy', np.array(best_thresh))
# print('Threshold disimpan: dataset2/xlmr_best_thresholds.npy')

# # Simpan .csv biar gampang dicek
# oof_df = pd.DataFrame(
#     full_probs,
#     columns=[f'prob_{c}' for c in label_encoder.classes_]
# )
# oof_df.insert(0, 'label', data_raw['label'])
# oof_df.insert(1, 'label_id', data_raw['label_id'])
# oof_df.insert(2, 'is_noisy', data_raw['is_noisy'])
# oof_df.to_csv('dataset2/xlmr_oof_proba.csv', index=False)
# print('OOF disimpan: dataset2/xlmr_oof_proba.csv')

# oof_df.head()

Predicting seluruh data asli untuk OOF...


OOF shape: (5000, 8)
Sum per baris (harus ~1): 1.0
OOF disimpan: dataset2/xlmr_oof_proba.npy
Threshold disimpan: dataset2/xlmr_best_thresholds.npy
OOF disimpan: dataset2/xlmr_oof_proba.csv


,label,label_id,is_noisy,prob_Anggaran,prob_Distribusi,prob_Ekonomi,prob_Kualitas Pangan,prob_Lainnya,prob_Politik,prob_Sasaran Penerima,prob_Tata Kelola
0,Sasaran Penerima,6,True,0.009339,0.016906,0.034583,0.885645,0.008660,0.006917,0.019078,0.018873
1,Politik,5,False,0.025213,0.003756,0.043182,0.004361,0.004517,0.895577,0.011125,0.012269
2,Sasaran Penerima,6,True,0.929616,0.007814,0.021285,0.004547,0.006633,0.006727,0.012973,0.010405
3,Sasaran Penerima,6,False,0.003223,0.003337,0.016437,0.002288,0.004815,0.005070,0.959681,0.005149
4,Politik,5,False,0.007595,0.005052,0.017389,0.004400,0.006846,0.935231,0.011344,0.012143


In [ ]:
df_test = pd.read_csv('dataset2/KasihLabel.csv')   # <-- ganti nama file
print('Shape:', df_test.shape)
print(df_test[['id', 'full_text']].head())

# ── Tokenisasi ───────────────────────────────────────────────
test_encodings = tokenizer(
    df_test['full_text'].tolist(),
    padding=True,
    truncation=True,
    max_length=128
)

# ── Dataset ──────────────────────────────────────────────────
test_dataset = SoftLabelDataset(
    test_encodings,
    [0] * len(df_test),   # dummy label, tidak dipakai
    soft_labels=None
)

# ── Prediksi ─────────────────────────────────────────────────
test_output = trainer.predict(test_dataset)
test_probs  = torch.softmax(
    torch.tensor(test_output.predictions), dim=-1
).numpy()


test_preds  = predict_with_threshold(test_probs, best_thresh)


df_test['label']      = [id2label[p] for p in test_preds]
df_test['confidence'] = test_probs[np.arange(len(test_preds)), test_preds].round(4)

print('\nDistribusi label hasil prediksi:')
print(df_test['label'].value_counts())
df_test[['id', 'full_text', 'label', 'confidence']].head(10)

Shape: (1500, 2)
        id                                          full_text
0  TXT0001  mbg di sekolah kota saya belum ada yang  dapat...
1  TXT0002  wkwkkdidaerah ku pun yang  dapat mbg baru bebe...
2  TXT0003  tahi  tahu tidak  ,gak semua sekolah dapat mbg...
3  TXT0004  gencar cegah stunting, mbg telah capai 49% sas...
4  TXT0005             140 siswa kupang keracunan program mbg



Distribusi label hasil prediksi:
label
Kualitas Pangan     345
Anggaran            243
Distribusi          200
Politik             179
Tata Kelola         166
Lainnya             163
Sasaran Penerima    150
Ekonomi              54
Name: count, dtype: int64


,id,full_text,label,confidence
0,TXT0001,mbg di sekolah kota saya belum ada yang dapat...,Distribusi,0.9037
1,TXT0002,wkwkkdidaerah ku pun yang dapat mbg baru bebe...,Distribusi,0.9036
2,TXT0003,"tahi tahu tidak ,gak semua sekolah dapat mbg...",Kualitas Pangan,0.8626
3,TXT0004,"gencar cegah stunting, mbg telah capai 49% sas...",Sasaran Penerima,0.9453
4,TXT0005,140 siswa kupang keracunan program mbg,Kualitas Pangan,0.8590
5,TXT0006,"mohon diloloskan min,, kejadian makan bergizi ...",Kualitas Pangan,0.8628
6,TXT0007,luar biasa putra bangsa miklos sunario berbica...,Politik,0.9104
7,TXT0008,kegagalan elu. bohong saja 19 juta lapangan ...,Ekonomi,0.6508
8,TXT0009,impor sapi demi mbg tuh yang di kasih apanya ...,Kualitas Pangan,0.8625
9,TXT0010,yang urus mbg kan nu..apalagi kejadian ini d...,Tata Kelola,0.9480


In [28]:
# ── Simpan hasil ─────────────────────────────────────────────
df_test.to_csv('dataset2/hasil_prediksi.xlsx', index=False)
print('done')

# file template-jawaban
df_output = df_test[['id', 'label']]
df_output.to_csv('dataset2/template-jawaban.csv', index=False)
print('done')

done
done
